In [7]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('https://raw.githubusercontent.com/alexeygrigorev/datasets/master/car_fuel_efficiency.csv')

In [3]:
df

,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,model_year,origin,fuel_type,drivetrain,num_doors,fuel_efficiency_mpg
0,170,3.0,159.0,3413.433759,17.7,2003,Europe,Gasoline,All-wheel drive,0.0,13.231729
1,130,5.0,97.0,3149.664934,17.8,2007,USA,Gasoline,Front-wheel drive,0.0,13.688217
2,170,NaN,78.0,3079.038997,15.1,2018,Europe,Gasoline,Front-wheel drive,0.0,14.246341
3,220,4.0,NaN,2542.392402,20.2,2009,USA,Diesel,All-wheel drive,2.0,16.912736
4,210,1.0,140.0,3460.870990,14.4,2009,Europe,Gasoline,All-wheel drive,2.0,12.488369
...,...,...,...,...,...,...,...,...,...,...,...
9699,140,5.0,164.0,2981.107371,17.3,2013,Europe,Diesel,Front-wheel drive,NaN,15.101802
9700,180,NaN,154.0,2439.525729,15.0,2004,USA,Gasoline,All-wheel drive,0.0,17.962326
9701,220,2.0,138.0,2583.471318,15.1,2008,USA,Diesel,All-wheel drive,-1.0,17.186587
9702,230,4.0,177.0,2905.527390,19.4,2011,USA,Diesel,Front-wheel drive,1.0,15.331551


# Q1

In [4]:
df.isna().sum()

engine_displacement      0
num_cylinders          482
horsepower             708
vehicle_weight           0
acceleration           930
model_year               0
origin                   0
fuel_type                0
drivetrain               0
num_doors              502
fuel_efficiency_mpg      0
dtype: int64

# Q2

In [5]:
df['horsepower'].describe()

count    8996.000000
mean      149.657292
std        29.879555
min        37.000000
25%       130.000000
50%       149.000000
75%       170.000000
max       271.000000
Name: horsepower, dtype: float64

In [6]:
df['horsepower'].median()

np.float64(149.0)

# Q3

In [8]:
# embaralhar com seed 42
n = len(df)
idx = np.arange(n)
rng = np.random.default_rng(42)
rng.shuffle(idx)

n_train = int(0.6*n)
n_val   = int(0.2*n)

train_idx = idx[:n_train]
val_idx   = idx[n_train:n_train+n_val]
test_idx  = idx[n_train+n_val:]

df_train = df.iloc[train_idx].copy()
df_val   = df.iloc[val_idx].copy()
df_test  = df.iloc[test_idx].copy()

# --- opção 1: imputar com 0
tr0 = df_train.fillna(0).copy()
va0 = df_val.fillna(0).copy()

X_tr0 = np.column_stack([
    np.ones(len(tr0)),
    tr0[['engine_displacement','horsepower','vehicle_weight','model_year']].values
])
y_tr0 = tr0['fuel_efficiency_mpg'].values

X_va0 = np.column_stack([
    np.ones(len(va0)),
    va0[['engine_displacement','horsepower','vehicle_weight','model_year']].values
])
y_va0 = va0['fuel_efficiency_mpg'].values

w0 = np.linalg.inv(X_tr0.T @ X_tr0) @ (X_tr0.T @ y_tr0)
pred_va0 = X_va0 @ w0
rmse0 = float(np.sqrt(np.mean((y_va0 - pred_va0)**2)))

# --- opção 2: imputar com média (usando apenas o treino)
means = df_train.mean(numeric_only=True)
trm = df_train.fillna(means).copy()
vam = df_val.fillna(means).copy()

X_trm = np.column_stack([
    np.ones(len(trm)),
    trm[['engine_displacement','horsepower','vehicle_weight','model_year']].values
])
y_trm = trm['fuel_efficiency_mpg'].values

X_vam = np.column_stack([
    np.ones(len(vam)),
    vam[['engine_displacement','horsepower','vehicle_weight','model_year']].values
])
y_vam = vam['fuel_efficiency_mpg'].values

wm = np.linalg.inv(X_trm.T @ X_trm) @ (X_trm.T @ y_trm)
pred_vam = X_vam @ wm
rmsem = float(np.sqrt(np.mean((y_vam - pred_vam)**2)))

print("RMSE com 0  :", round(rmse0, 2))
print("RMSE com média:", round(rmsem, 2))

# Esperado: "With mean" (média melhor que 0).


RMSE com 0  : 0.52
RMSE com média: 0.47


# Q4

In [9]:
# Reutilizar os splits da célula anterior (df_train, df_val)
tr = df_train.fillna(0).copy()
va = df_val.fillna(0).copy()

X_tr = np.column_stack([
    np.ones(len(tr)),
    tr[['engine_displacement','horsepower','vehicle_weight','model_year']].values
])
y_tr = tr['fuel_efficiency_mpg'].values

X_va = np.column_stack([
    np.ones(len(va)),
    va[['engine_displacement','horsepower','vehicle_weight','model_year']].values
])
y_va = va['fuel_efficiency_mpg'].values

r_list = [0, 0.01, 0.1, 1, 5, 10, 100]
rmse_by_r = {}

for r in r_list:
    XTX = X_tr.T @ X_tr
    if r > 0:
        XTX = XTX + r * np.eye(XTX.shape[0])
    w = np.linalg.inv(XTX) @ (X_tr.T @ y_tr)
    pred = X_va @ w
    rmse_r = float(np.sqrt(np.mean((y_va - pred)**2)))
    rmse_by_r[r] = round(rmse_r, 2)

print("RMSE por r:", rmse_by_r)
best_r = min(rmse_by_r, key=lambda k: (rmse_by_r[k], k))  # menor RMSE; em empate, menor r
print("Melhor r:", best_r)


RMSE por r: {0: 0.52, 0.01: 0.52, 0.1: 0.52, 1: 0.53, 5: 0.53, 10: 0.53, 100: 0.53}
Melhor r: 0


# Q5

In [10]:
rmses = []

for seed in range(10):
    # split por seed
    n = len(df)
    idx = np.arange(n)
    rng = np.random.default_rng(seed)
    rng.shuffle(idx)

    n_train = int(0.6*n)
    n_val   = int(0.2*n)

    train_idx = idx[:n_train]
    val_idx   = idx[n_train:n_train+n_val]

    tr = df.iloc[train_idx].copy().fillna(0)
    va = df.iloc[val_idx].copy().fillna(0)

    X_tr = np.column_stack([
        np.ones(len(tr)),
        tr[['engine_displacement','horsepower','vehicle_weight','model_year']].values
    ])
    y_tr = tr['fuel_efficiency_mpg'].values

    X_va = np.column_stack([
        np.ones(len(va)),
        va[['engine_displacement','horsepower','vehicle_weight','model_year']].values
    ])
    y_va = va['fuel_efficiency_mpg'].values

    # OLS
    w = np.linalg.inv(X_tr.T @ X_tr) @ (X_tr.T @ y_tr)
    pred = X_va @ w
    rmse_seed = float(np.sqrt(np.mean((y_va - pred)**2)))
    rmses.append(round(rmse_seed, 2))  # segue enunciado: arredondar RMSEs a 2 casas

rmses = np.array(rmses, dtype=float)
std_scores = float(np.std(rmses))
print("RMSEs por seed:", rmses)
print("Desvio-padrão (np.std), 3 casas:", round(std_scores, 3))

RMSEs por seed: [0.52 0.52 0.53 0.52 0.53 0.53 0.52 0.51 0.52 0.53]
Desvio-padrão (np.std), 3 casas: 0.006


# Q6

In [11]:
# split com seed=9
n = len(df)
idx = np.arange(n)
rng = np.random.default_rng(9)
rng.shuffle(idx)

n_train = int(0.6*n)
n_val   = int(0.2*n)

train_idx = idx[:n_train]
val_idx   = idx[n_train:n_train+n_val]
test_idx  = idx[n_train+n_val:]

train9 = df.iloc[train_idx].copy()
val9   = df.iloc[val_idx].copy()
test9  = df.iloc[test_idx].copy()

# juntar treino+validação
train_full = pd.concat([train9, val9], ignore_index=True)

# imputar 0
tr = train_full.fillna(0).copy()
te = test9.fillna(0).copy()

# preparar matrizes
X_tr = np.column_stack([
    np.ones(len(tr)),
    tr[['engine_displacement','horsepower','vehicle_weight','model_year']].values
])
y_tr = tr['fuel_efficiency_mpg'].values

X_te = np.column_stack([
    np.ones(len(te)),
    te[['engine_displacement','horsepower','vehicle_weight','model_year']].values
])
y_te = te['fuel_efficiency_mpg'].values

# Ridge com r=0.001
r = 0.001
XTX = X_tr.T @ X_tr + r * np.eye(X_tr.shape[1])
w = np.linalg.inv(XTX) @ (X_tr.T @ y_tr)

pred_te = X_te @ w
rmse_test = float(np.sqrt(np.mean((y_te - pred_te)**2)))
print("RMSE teste (seed=9, r=0.001):", round(rmse_test, 3))

# Esperado: 5.15


RMSE teste (seed=9, r=0.001): 0.505
